# 06 · Camada Silver
Este notebook lê a tabela `bronze_reclamacoes` e aplica os tratamentos necessários para gerar a tabela `silver_reclamacoes`. A primeira etapa é a correção do schema: padronização dos nomes das colunas e conversão dos tipos de dados.


In [0]:
from pyspark.sql import functions as F


bronze = spark.table("workspace.consumidor.bronze_reclamacoes")
bronze.printSchema()

## 7. Correção do tipos no schema



In [0]:
# Dicionário "nome antigo": "nome novo"
novos_nomes = {
    "Região": "regiao",
    "UF": "uf",
    "Cidade": "cidade",
    "Sexo": "sexo",
    "Faixa Etária": "faixa_etaria",
    "Data Finalização": "data_finalizacao",
    "Tempo Resposta": "tempo_resposta",
    "Nome Fantasia": "nome_fantasia",
    "Segmento de Mercado": "segmento_mercado",
    "Área": "area",
    "Assunto": "assunto",
    "Grupo Problema": "grupo_problema",
    "Problema": "problema",
    "Como Comprou Contratou": "como_comprou_contratou",
    "Procurou Empresa": "procurou_empresa",
    "Respondida": "respondida",
    "Situação": "situacao",
    "Avaliação Reclamação": "avaliacao_reclamacao",
    "Nota do Consumidor": "nota_consumidor"
}

silver = bronze.withColumnsRenamed(novos_nomes)
print(silver.columns)
display(silver)

In [0]:
display(silver.select("data_finalizacao"))

In [0]:
from pyspark.sql.functions import col
from pyspark.sql.types import DateType



silver = silver.withColumn("data_finalizacao", col("data_finalizacao").cast(DateType()))

In [0]:
silver.printSchema()

## 8. Filtro de Bancos, Financeiras e Administradoras de Cartão na coluna de segmentos de mercado

In [0]:
display(silver.filter(col("segmento_mercado") == "Bancos, Financeiras e Administradoras de Cartão"))
    

745 apenas com notas de avaliação na categoria Bancos


In [0]:

# Mantém apenas o segmento bancário, foco das perguntas de negócio
silver = silver.filter(col("segmento_mercado") == "Bancos, Financeiras e Administradoras de Cartão")
print("Reclamações do segmento bancário:", silver.count())

In [0]:
# Aplica trim em todas as colunas do tipo texto
for nome, tipo in silver.dtypes:
    if tipo == "string":
        silver = silver.withColumn(nome, F.trim(col(nome)))

display(silver.groupBy("regiao").count())

In [0]:
# Conta os nulos de cada coluna
display(silver.select([F.sum(col(c).isNull().cast("int")).alias(c) for c in silver.columns]))

# Preenche sexo vazio
silver = silver.fillna({"sexo": "Não informado"})

## 9. Registros duplicados
Foram identificadas linhas idênticas em todas as colunas originais. Como a base não possui um identificador único por reclamação, considerou-se que esses registros representam duplicidades na publicação dos dados, e eles foram removidos para evitar que a mesma reclamação fosse contada mais de uma vez nas análises.

In [0]:
# Colunas originais (sem as de controle, que variam entre cargas)
colunas_originais = [c for c in silver.columns if c not in ("arquivo_origem", "data_carga")]

# Contagem antes da remoção
antes = silver.count()

# Remove as linhas idênticas, mantendo uma de cada
silver = silver.dropDuplicates(colunas_originais)

# Contagem depois da remoção
depois = silver.count()
print("Linhas antes:", antes)
print("Linhas depois:", depois)
print("Duplicadas removidas:", antes - depois)

## 10. Salvar a finalização da Silver

In [0]:
silver.write.mode("overwrite").saveAsTable("workspace.consumidor.silver_reclamacoes")

tabela_silver = spark.table("workspace.consumidor.silver_reclamacoes")
print("Linhas na silver:", tabela_silver.count())
tabela_silver.printSchema()